# **Coeficiente de localización Baja California Sur**

El coeficiente de localización (LQ) es un indicador estadístico que permite evaluar qué tan concentrada está una industria, ocupación o grupo demográfico determinado en una zona geográfica concreta, la cual toma como referencia una región de mayor tamaño que suele ser nacional o regional.

Para el siguiente análisis el espacio demográfico determinado es el estado de Baja California Sur y como región de referencia de mayor tamaño el país.

El análisis busca responder a las siguientes preguntas: 

1. ¿En qué sectores destaca laboralmente el estado?
2. ¿La economía estatal se diversificó o concentro?
3. ¿Qué sectores podrían abastecer una demanda fuera del estado?
4. ¿El estado se hizo más especializado en algún sector?
5. ¿El estado se hizo menos especializado en algún sector?

# **Fuente**

- **Fuente:** INEGI, Censos Económicos 2024.

Ver `README.md` para el detalle completo de la consulta y notas metodológicas.

# **Código y resultados**

In [1]:
# Importar librería

import pandas as pd

# Cargar datos

datos_bcs = pd.read_csv('../data/datos_bcs.csv')

# Renombrar columnas

datos_bcs = datos_bcs.rename(columns={
    'Año Censal': 'año',
    'Entidad': 'entidad',
    'Actividad económica': 'sector',
    'H001A Personal ocupado total': 'pot'
})

In [2]:
# Función para calcular el coeficiente de localización por año

def coeficiente_localizacion(datos, año, entidad_estado, entidad_nacional):
    
    # 1. Filtrar por año
    
    df_año = datos.query('año == @año')
    
    # 2. Separar en dos dataframes nacional y estado
    
    nacional = df_año.query('entidad == @entidad_nacional')[['sector', 'pot']]
    estado = df_año.query('entidad == @entidad_estado')[['sector', 'pot']]
    
    # 3. Renombrar columnas de personal ocupado en ambos dataframe por pot_nacional y pot_bcs
    
    nacional = nacional.rename(columns={'pot': 'pot_nacional'})
    estado = estado.renae(columns={'pot': 'pot_bcs'})
    
    # 4. Combinar ambos dataframes mediante sector
    
    df_bcs = pd.merge(nacional, estado, on='sector', how='left')
    
    # 5. Calcular total_nacional y total_bcs (total del personal ocupado)
    
    total_nacional = df_bcs['pot_nacional'].sum()
    total_bcs = df_bcs['pot_bcs'].sum()
    
    # 6. Calcular LQ, empleo_esperado u empleo_excedente
    
    df_bcs['LQ'] = (df_bcs['pot_bcs'] / total_bcs) / (df_bcs['pot_nacional'] / total_nacional)
    df_bcs['empleo_esperado'] = (df_bcs['pot_nacional'] / total_nacional) * total_bcs
    df_bcs['empleo_excedente'] = df_bcs['pot_bcs'] - df_bcs['empleo_esperado']
    
    return df_bcs.set_index('sector')

In [ ]:
# Ingresar datos a la función y obtener resultados

# 1. Crear lista de años a análizar y diccionario para guardar resultados

# 1.1. Lista de años

lista_años = [2008, 2013, 2018, 2023]

# 1.2. Diccionario de resultados

resultados = {}

# 2. Bucle for para ingresar datos a la función

for año in lista_años:
    resultados[año] = coeficiente_localizacion(datos_bcs, año, '03 Baja California Sur', '00 Total Nacional')